In [3]:
# 2.1 Import Library dan Konfigurasi Path
import pandas as pd
import numpy as np
import os

# Konfigurasi path
DATA_PATH = '../../../datapreprocessingcopy/data_preprocessing_final.csv'
SLA_LEXICON_PATH = '../../outputs/SLA/sla_lexicon_adapted.csv' # Memuat leksikon yang sudah dibuat
OUTPUT_DIR = '../../outputs/SLA'

os.makedirs(OUTPUT_DIR, exist_ok=True)
print("[INFO] Library dan konfigurasi path berhasil dimuat.")

[INFO] Library dan konfigurasi path berhasil dimuat.


In [4]:
# 2.2 Load Data Preprocessing Final
df = pd.read_csv(DATA_PATH)

print(f"\nData preprocessing berhasil dimuat: {len(df)} tweet")
print(f"Kolom: {df.columns.tolist()}")
df.head()


Data preprocessing berhasil dimuat: 13192 tweet
Kolom: ['no', 'timestamp', 'teks', 'teks_processed']


,no,timestamp,teks,teks_processed
0,1,2016-12-30T06:37:56.000Z,ADIL loh utk yg punya kebijakan publik negara ...,ADIL loh untuk yang punya kebijakan publik neg...
1,2,2016-12-30T06:30:36.000Z,Tertibkan Media Online DPR Pemerintah Jangan S...,tertib media online DPR pemerintah jangan spor...
2,3,2016-12-30T04:48:35.000Z,harus dievaluasi lg kebijakan bebas visa truta...,harus evaluasi lagi kebijakan bebas visa utama...
3,4,2016-12-30T04:21:40.000Z,jangan ngambang aturan logis apa undang undang,jangan ngambang pengaturan logis apa undang un...
4,5,2016-12-30T02:36:13.000Z,Kebebasan bersuara berpendapat memang dijamin ...,bebas suara dapat memang jamin UU tetapi bebas...


In [5]:
# 2.3 Load Leksikon SLA yang Telah Diadaptasi
df_sla = pd.read_csv(SLA_LEXICON_PATH)
df_sla['kata'] = df_sla['kata'].astype(str).str.strip().str.lower()

# Dictionary lexicon untuk matching (SLA menggunakan kolom 'mean' sebagai skor)
sla_dict = dict(zip(df_sla['kata'], df_sla['mean']))

print(f"[INFO] Leksikon SLA berhasil dimuat: {len(sla_dict)} entri.")

[INFO] Leksikon SLA berhasil dimuat: 9071 entri.


In [7]:
# 2.4 Definisi Kategori Kata Fungsi
NEGASI_DAN_MODAL = {
    'tidak', 'bukan', 'jangan', 'belum', 'sangat', 'harus', 'wajib',
    'akan', 'sudah', 'sedang', 'telah', 'boleh', 'bisa'
}
KATA_HUBUNG_PREPOSISI = {
    'dan', 'atau', 'tetapi', 'karena', 'jika', 'di', 'ke', 'dari',
    'pada', 'untuk', 'dengan', 'oleh', 'hingga', 'sejak'
}
PRONOMINA_DEMONSTRATIVA = {
    'saya', 'aku', 'dia', 'kami', 'kamu', 'anda', 'ini', 'itu', 'yang'
}
PARTIKEL_KATA_TANYA = {
    'pun', 'sih', 'ya', 'lah', 'kah', 'apa', 'siapa', 'bagaimana'
}

ALL_FUNCTION_WORDS = (
    NEGASI_DAN_MODAL
    | KATA_HUBUNG_PREPOSISI
    | PRONOMINA_DEMONSTRATIVA
    | PARTIKEL_KATA_TANYA
)

In [9]:
# 2.5 Konfigurasi Remove Set (Hanya kata fungsi non-heuristik yang dihapus fisik)
REMOVE_SET = KATA_HUBUNG_PREPOSISI | PRONOMINA_DEMONSTRATIVA | PARTIKEL_KATA_TANYA

# Diagnostik: Mengecek kata fungsi di dalam Leksikon SLA
# Gunakan df_sla yang sudah memuat mean dan std hasil adaptasi
df_found_sla = df_sla[df_sla['kata'].isin(ALL_FUNCTION_WORDS)].copy().sort_values('kata')

# Ambil hanya yang benar-benar ada di Leksikon SLA untuk dihapus
remove_set_final = set(df_found_sla[df_found_sla['kata'].isin(REMOVE_SET)]['kata'])

print(f"[DIAGNOSTIK] Total kata fungsi ditemukan di leksikon SLA: {len(df_found_sla)}")
print(f"[KONFIGURASI] Kata fungsi yang akan dihapus dari aliran teks: {len(remove_set_final)} kata.")

# Menampilkan daftar kata yang akan dihapus
print("\n[DAFTAR] Kata yang dihapus fisik:")
print(sorted(list(remove_set_final)))

[DIAGNOSTIK] Total kata fungsi ditemukan di leksikon SLA: 22
[KONFIGURASI] Kata fungsi yang akan dihapus dari aliran teks: 13 kata.

[DAFTAR] Kata yang dihapus fisik:
['aku', 'anda', 'apa', 'dari', 'dia', 'itu', 'karena', 'pada', 'pun', 'saya', 'siapa', 'ya', 'yang']


In [10]:
# 2.6 Fungsi Tokenisasi
def tokenize(text):
    if not isinstance(text, str):
        return []
    return text.split()

df['tokens'] = df['teks_processed'].apply(tokenize)
print(f"\n[INFO] Tokenisasi selesai. Total token: {df['tokens'].str.len().sum():,}")


[INFO] Tokenisasi selesai. Total token: 235,560


In [ ]:
# 2.7 Fungsi Lexicon Matching dengan Penghapusan Kata Fungsi
def match_lexicon_sla_remove(tokens, lexicon, remove_set):
    matched, unmatched, removed = [], [], []
    for token in tokens:
        t_low = token.lower()
        if t_low in remove_set: removed.append(token)
        elif t_low in lexicon: matched.append(token)
        else: unmatched.append(token)
    return matched, unmatched, removed

# Terapkan ke DataFrame
df[['matched_words', 'unmatched_words', 'removed_words']] = pd.DataFrame(
    df['tokens'].apply(lambda x: match_lexicon_sla_remove(x, sla_dict, REMOVE_SET)).tolist(),
    index=df.index
)

In [13]:
# 2.8 Penerapan Lexicon Matching
print("\n[PROSES] Menjalankan lexicon matching SLA dengan penghapusan fungsi...")

df[['matched_words', 'removed_words', 'unmatched_words']] = pd.DataFrame(
    df['tokens'].apply(lambda x: match_lexicon_sla_remove(x, sla_dict, REMOVE_SET)).tolist(),
    index=df.index
)

print("[INFO] Lexicon matching selesai.")


[PROSES] Menjalankan lexicon matching SLA dengan penghapusan fungsi...
[INFO] Lexicon matching selesai.


In [17]:
# 2.9 Perhitungan Statistik
total_words = df['tokens'].str.len().sum()
total_matched = df['matched_words'].str.len().sum()
total_removed = df['removed_words'].str.len().sum()
total_unmatched = df['unmatched_words'].str.len().sum()

# Token sisa setelah penghapusan (konten + negasi/modal)
filtered_words = total_matched + total_unmatched

print("\n[STATISTIK] Hasil Lexicon Matching (SLA + Hapus Fungsi):")
print(f"Total token awal         : {total_words:,}")
print(f"Dihapus (kata fungsi)    : {total_removed:,} ({(total_removed/total_words)*100:.2f}%)")
print(f"Token sisa (konten)      : {filtered_words:,}")
print(f"Matched di SLA           : {total_matched:,} ({(total_matched/filtered_words)*100:.2f}% dari token sisa)")
print(f"Unmatched                : {total_unmatched:,} ({(total_unmatched/filtered_words)*100:.2f}% dari token sisa)")
print(f"Coverage Rate (konten)   : {(total_matched/filtered_words)*100:.2f}%")


[STATISTIK] Hasil Lexicon Matching (SLA + Hapus Fungsi):
Total token awal         : 235,560
Dihapus (kata fungsi)    : 128,260 (54.45%)
Token sisa (konten)      : 107,300
Matched di SLA           : 83,100 (77.45% dari token sisa)
Unmatched                : 24,200 (22.55% dari token sisa)
Coverage Rate (konten)   : 77.45%


In [18]:
# 2.10 Preview Hasil Matching
print("\n[PREVIEW] 3 Tweet Pertama:")
for i in range(3):
    print(f"\nTweet {i+1}: {df['teks_processed'].iloc[i][:80]}...")
    print(f"  Matched  : {df['matched_words'].iloc[i][:5]}")
    print(f"  Removed  : {df['removed_words'].iloc[i][:5]}")
    print(f"  Unmatched: {df['unmatched_words'].iloc[i][:5]}")


[PREVIEW] 3 Tweet Pertama:

Tweet 1: ADIL loh untuk yang punya kebijakan publik negara ingat yang ini ! !...
  Matched  : ['ADIL', 'punya', 'kebijakan', 'ingat']
  Removed  : ['loh', 'publik', 'negara', '!', '!']
  Unmatched: ['untuk', 'yang', 'yang', 'ini']

Tweet 2: tertib media online DPR pemerintah jangan sporadis apalagi selektif hanya kepada...
  Matched  : ['tertib', 'jangan', 'sporadis', 'selektif', 'hanya']
  Removed  : ['media', 'online', 'DPR', 'pemerintah', 'apalagi']
  Unmatched: ['yang']

Tweet 3: harus evaluasi lagi kebijakan bebas visa utama untuk negara tiongkok pak ! ! bah...
  Matched  : ['harus', 'lagi', 'kebijakan', 'bebas', 'bahaya']
  Removed  : ['evaluasi', 'visa', 'utama', 'negara', 'tiongkok']
  Unmatched: ['untuk']


In [19]:
# 2.11 Simpan Output
output_path = os.path.join(OUTPUT_DIR, 'sla_lexicon_matching_remove_func.csv')
df.to_csv(output_path, index=False)
print(f"\n[OUTPUT] Data berhasil disimpan ke: {output_path}")


[OUTPUT] Data berhasil disimpan ke: ../../outputs/SLA\sla_lexicon_matching_remove_func.csv
